# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [22]:
! pip install -q schedule pytest # установка библиотек, если ещё не

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [23]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [24]:
from re import search
from typing import Any
from logging import getLogger

from bs4 import BeautifulSoup
from requests import get as requests_get
from requests.exceptions import (
    ConnectionError as RequestsConnectionError,
    HTTPError,
    RequestException,
    Timeout,
)

BASE_URL = "https://books.toscrape.com/"
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}


logger = getLogger(__name__)


def get_book_data(book_url: str) -> dict[str, Any] | None:
    """
    Получает информацию о книге с указанной страницы.

    Функция состоит из двух этапов — сетевого запроса и разбора
    HTML.  Все ожидаемые ошибки этих этапов перехватываются,
    логируются и приводят к возврату ``None``.  Неожидаемые
    исключения логируются через ``logger.exception`` и
    пробрасываются наружу, чтобы не скрывать баги.

    Parameters
    ----------
    book_url : str
        Абсолютный URL страницы книги на books.toscrape.com.

    Returns
    -------
    dict[str, Any] | None
        Словарь с данными о книге при успехе. Ключи:

        - ``"url"`` (str) — исходный URL книги;
        - ``"title"`` (str) — название (пустая строка, если не найдено);
        - ``"price"`` (str) — цена как в HTML, например ``"£51.77"``;
        - ``"rating"`` (int) — числовой рейтинг (0–5) по ``RATING_MAP``;
        - ``"rating_text"`` (str) — словесный рейтинг (``"One"``…
          ``"Five"``), ``"Zero"`` если не найден;
        - ``"availability"`` (str) — текст блока наличия;
        - ``"quantity"`` (int) — число доступных экземпляров
          (0, если не удалось распарсить);
        - ``"description"`` (str) — описание (пустая строка, если
          отсутствует);
        - ``"product_information"`` (dict[str, str]) — таблица
          Product Information (пустой словарь, если таблицы нет).

        ``None`` возвращается, если страницу не удалось получить
        или разобрать:

        - сетевые ошибки запроса: таймаут, обрыв соединения,
          SSL/прокси-ошибки и прочие ``RequestException``;
        - HTTP-ошибки ``4xx``/``5xx``;
        - некорректная или нестандартная структура HTML: отсутствует
          ``div.product_main``, ожидаемые теги имеют неожидаемый тип
          (``ValueError``, ``AttributeError``, ``IndexError``,
          ``KeyError``).

    Raises
    ------
    Exception
        Любое исключение, не относящееся к перечисленным выше
        ожидаемым категориям, логируется и пробрасывается
        вызывающему коду — чтобы не маскировать реальные
        программные ошибки.

    Notes
    -----
    Таймаут сетевого запроса зафиксирован на 15 секундах.
    """
    logger.info("Get data from url %s", book_url)

    # Запрос к ресурсу
    try:
        response = requests_get(book_url, timeout=15)
        response.raise_for_status()
    except HTTPError as error:
        status = getattr(error.response, "status_code", "unknown")
        logger.error(
            "HTTP error while fetching %s (status=%s): %s",
            book_url,
            status,
            error,
        )
        return None
    except Timeout:
        logger.error("Timeout while fetching %s (timeout=15s)", book_url)
        return None
    except RequestsConnectionError as error:
        logger.error("Connection error while fetching %s: %s", book_url, error)
        return None
    except RequestException as error:
        # catch-all for other requests failures (SSL, proxy, too many redirects, …)
        logger.error("Request error while fetching %s: %s", book_url, error)
        return None

    # Парсинг
    try:
        soup = BeautifulSoup(response.text, "html.parser")

        product_main = soup.find("div", class_="product_main")
        if product_main is None:
            raise ValueError(f"Не найден блок product_main: {book_url}")

        title_tag = product_main.find("h1")
        price_tag = product_main.find("p", class_="price_color")
        rating_tag = product_main.find("p", class_="star-rating")
        availability_tag = product_main.find("p", class_="instock availability")

        title = title_tag.get_text(strip=True) if title_tag else ""
        price = price_tag.get_text(strip=True) if price_tag else ""

        rating_classes = rating_tag.get("class", []) if rating_tag else []
        rating_word = rating_classes[-1] if rating_classes else "Zero"
        rating = RATING_MAP.get(rating_word, 0)

        availability = (
            availability_tag.get_text(" ", strip=True) if availability_tag else ""
        )

        quantity_match = search(r"\((\d+)\s+available\)", availability)
        quantity = int(quantity_match.group(1)) if quantity_match else 0

        description = ""
        description_tag = soup.find("div", id="product_description")
        if description_tag:
            description_paragraph = description_tag.find_next_sibling("p")
            if description_paragraph:
                description = description_paragraph.get_text(" ", strip=True)

        product_information: dict[str, str] = {}
        table = soup.find("table", class_="table-striped")
        if table:
            for row in table.find_all("tr"):
                th = row.find("th")
                td = row.find("td")
                if th and td:
                    product_information[th.get_text(strip=True)] = td.get_text(
                        " ", strip=True
                    )

        return {
            "url": book_url,
            "title": title,
            "price": price,
            "rating": rating,
            "rating_text": rating_word,
            "availability": availability,
            "quantity": quantity,
            "description": description,
            "product_information": product_information,
        }

    except (ValueError, AttributeError, IndexError, KeyError) as error:
        # Страница имеет нестандартную структуру
        logger.error("Failed to parse book page %s: %s", book_url, error)
        return None
    except Exception:
        # Обработка неожидаемых исключений. Выбрасываем исключение,
        # так как логичнее делегировать обработку таких ошибок вызывающей
        # функции
        logger.exception("Unexpected error while processing %s", book_url)
        raise

In [25]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'url': 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
 'title': 'A Light in the Attic',
 'price': 'Â£51.77',
 'rating': 3,
 'rating_text': 'Three',
 'availability': 'In stock (22 available)',
 'quantity': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and 

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [26]:
from json import dumps
from logging import getLogger
from urllib.parse import urljoin
from pathlib import Path

from requests import get as requests_get
from requests.exceptions import HTTPError, RequestException
from bs4 import BeautifulSoup

"""Парсер книг с сайта books.toscrape.com."""


logger = getLogger(__name__)

BASE_URL = "https://books.toscrape.com/"
START_URL = urljoin(BASE_URL, "catalogue/page-1.html")

logger = getLogger(__name__)


def scrape_books(
    start_url: str = START_URL,
    is_save: bool = True,
    append: bool = True,
    max_pages: int | None = None,
    output_path: str | Path = "../artifacts/books_data.txt",
) -> int:
    """
    Собирает данные со всех страниц каталога books.toscrape.com
    и пишет их в файл построчно (JSONL), не накапливая
    в памяти.

    Файл открывается в режиме ``"a"`` (append), поэтому
    создаётся автоматически, если его не существует. При
    ``append=False`` файл перезаписывается. Родительские
    папки создаются автоматически.

    Ошибки на уровне отдельной книги (недоступная страница,
    некорректный HTML, отсутствующие поля) не прерывают обход:
    книга пропускается с записью в лог, обработка продолжается
    со следующей. 404 на страницах каталога, начиная со второй,
    трактуется как конец каталога.

    Parameters
    ----------
    start_url : str, default=START_URL
        URL первой страницы каталога.
    is_save : bool, default=True
        Если True — книги пишутся в ``output_path``.
        Если False — данные не сохраняются, но обход и подсчёт
        книг выполняются (полезно для «сухого прогона» /
        проверки доступности).
    append : bool, default=True
        True — дописывать в конец файла (``"a"``),
        False — перезаписывать (``"w"``).
    max_pages : int | None, default=None
        Максимальное количество страниц каталога, которые нужно
        обойти (имеются в виду именно страницы каталога, а не
        книги). ``None`` — без ограничения. Например,
        ``max_pages=1`` обработает только первую страницу
        каталога.
    output_path : str | Path, default="books_data.txt"
        Путь к файлу. Родительские папки создаются
        автоматически.

    Returns
    -------
    int
        Количество успешно обработанных книг (независимо от
        значения ``is_save``).

    Raises
    ------
    HTTPError
        Не-404 ошибка при загрузке страницы каталога, либо 404
        на первой странице каталога (``page_number == 1``).
    RequestException
        Сетевые ошибки при загрузке страницы каталога:
        таймаут, обрыв соединения и т.п.

    Notes
    -----
    Ошибки загрузки/разбора страницы отдельной книги (в том
    числе непойманные исключения из :func:`get_book_data`)
    не пробрасываются наружу — они логируются как warning,
    а соответствующая книга пропускается и не увеличивает
    возвращаемый счётчик.
    """
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    mode = "a" if append else "w"

    count = 0
    page_url: str | None = start_url
    page_number = 1

    # Файл открываем один раз; open("a") сам создаст его, если нет.
    with output_file.open(mode, encoding="utf-8") as file:
        while page_url:
            # Если номер страницы больше, чем максимальное число страниц
            # прерываем выполнение цикла
            if max_pages is not None and page_number > max_pages:
                break

            try:
                logger.info("Запрос к странице: %s", page_url)

                response = requests_get(page_url, timeout=15)
                response.raise_for_status()
            except HTTPError as error:
                status = (
                    error.response.status_code if error.response is not None else None
                )

                if status == 404 and page_number > 1:
                    logger.info("Достигнут конец каталога (404): %s", page_url)
                    break

                if status == 404:
                    logger.error(
                        "Стартовая страница каталога недоступна (404): %s",
                        page_url,
                    )
                    raise

                logger.error(
                    "Ошибка загрузки каталога %s (status=%s): %s",
                    page_url,
                    status,
                    error,
                )
                raise
            except RequestException as error:
                logger.error("Сетевая ошибка на %s: %s", page_url, error)
                raise

            soup = BeautifulSoup(response.text, "html.parser")

            for article in soup.select("article.product_pod"):
                link = article.select_one("h3 a")
                if link is None or not link.get("href"):
                    continue

                book_url = urljoin(page_url, link["href"])
                try:
                    book = get_book_data(book_url)
                    if book is None:
                        logger.warning(
                            "Не удалось получить данные для книги по адресу, %s",
                            book_url,
                        )
                        continue
                except Exception as error:
                    logger.warning("Пропущена книга %s: ; Причина: %s", book_url, error)
                    continue

                if is_save:
                    file.write(dumps(book, ensure_ascii=False) + "\n")
                count += 1

            next_link = soup.select_one("li.next a")
            page_url = urljoin(page_url, next_link["href"]) if next_link else None

            if page_url is None:
                # Если страницы закончились прерываем цикл
                break

            page_number += 1

    if is_save:
        logger.info(
            "Данные сохранены: %s (всего книг: %d)",
            output_file.resolve(),
            count,
        )
    else:
        logger.info(
            "Обработано адресов: %d)",
            count,
        )

In [30]:
# Проверка работоспособности функции
# Что бы не переполнять память, так как в теории книг может быть очень много,
# настроил запись книг в файл построчно
scrape_books(is_save=True, max_pages=1)  # Допишите ваши аргументы

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
from time import strftime
from schedule import clear, every, run_pending


def scheduled_scrape() -> None:
    """Запускает сбор данных и сохраняет результат."""
    books = scrape_books(is_save=True, max_pages=1)
    current_time = strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{current_time}] Собрано и сохранено книг: " f"{len(books)}")


def start_scheduler(run_time: str = "19:00") -> None:
    """
    Запускает бесконечный планировщик.

    Parameters
    ----------
    run_time : str, default="19:00"
        Время ежедневного запуска в формате HH:MM.
    """
    clear()
    every().day.at(run_time).do(scheduled_scrape)

    print(f"Планировщик запущен. " f"Сбор данных ежедневно в {run_time}.")

    try:
        while True:
            run_pending()
            time.sleep(60)
    except KeyboardInterrupt:
        print("Планировщик остановлен.")


# Основной запуск:
# start_scheduler("19:00")

# Локальная проверка на ближайшем времени:
# start_scheduler("14:35")

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [29]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest -v ../tests

============================= test session starts =============================
platform win32 -- Python 3.12.2, pytest-9.2.0, pluggy-1.7.0 -- E:\propgram\ml\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: e:\propgram\ml\mfti\books_scrapper
configfile: pytest.ini
collecting ... collected 8 items

..\tests\test_get_book_data.py::test_get_book_data_parses_full_page PASSED [ 12%]
..\tests\test_get_book_data.py::test_get_book_data_fills_defaults_when_optional_fields_are_missing PASSED [ 25%]
..\tests\test_get_book_data.py::test_get_book_data_returns_none_on_http_error PASSED [ 37%]
..\tests\test_imports.py::test_package_exports_public_api PASSED         [ 50%]
..\tests\test_imports.py::test_package_modules_import PASSED             [ 62%]
..\tests\test_scrapper.py::test_scrape_books_writes_parsed_book PASSED   [ 75%]
..\tests\test_scrapper.py::test_scrape_books_does_not_request_pages_past_max_pages PASSED [ 87%]
..\tests\test_scrapper.py::test_scrape_books_raises_when_first_catal

## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: выполнить задание по получению данных с сайта с тестовыми данными и загрузка проекта в GIT. Получилось всё.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Некоторые затруднения возникли с библиотекой beautiful soup, так как раньше с ней не работал. В остальном, подобные задачи я уже выполнял.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Написал моки для тестов(справедливости ради не помню, что бы кто-то из моих коллег или я сам писали тесты руками вне обучения)`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Вопросов нет. Задание пределоьно понятное.`

**5. Самооценка по критериям задания**:

`Ответ: Всё получилось.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Ниего не могу сказать про доработки. Задание показалось интересным, было полезно поработать с библиотекой beautiful soup. По поводу библиотеки requests: в силу специфики фреймворка используеммго на работе. в основном использую aiohttp И httpx, так что было полезно освежить знание requests.`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.
